# RSVT — M3Dsynth Training and Results (coarse-to-fine)

This notebook prepares the **M3Dsynth** dataset (manipulated LIDC-IDRI lung CT: nodule removal and insertion by CycleGAN, pix2pix and diffusion generators), checks it, trains the coarse-to-fine RSVT and shows its results.

**Just Run all; the defaults are ready.** Each run decides what to do from what is already saved in Drive:

1. **First, on a CPU runtime** (cheapest): it downloads and verifies the dataset, runs the data checks and stops before training. This takes hours; keep the tab open.
2. **Then on a GPU runtime** (Runtime → Change runtime type → L4): it reads the saved dataset back, reuses the checks and trains.
3. **After a disconnect**, Run all again: it continues the unfinished training run with the code it started with.

When a run finishes and every saved file has been read back from Drive, the last cell releases the runtime so it stops using compute units; the outputs stay visible.

The project code is cloned from GitHub, and the exact commit is saved with each run. Every output saves automatically to project Drive and is read back against a checksum.

With the default patient caps the download is about 300 LIDC-IDRI series from the public TCIA API (~33 GB) plus one pass over the three author archives (cycle 171 GB, pix2pix 142 GB, diffusion 161 GB) from grip.unina.it. Only verified boxes around each edit are kept, so Drive storage stays near 25 GB (boxes kept from an earlier rule come on top). A dropped connection reconnects at the same byte; if the runtime restarts, finished archives and samples are kept and only the current archive is read again.

Licences: M3Dsynth (GRIP-UNINA, non-profit research use with attribution) and LIDC-IDRI (TCIA, CC BY 3.0). Test patients are not downloaded or opened here.


## 1. Choose the run settings

Why: everything you might change is here, and the defaults are ready to run. There are no stage switches: the notebook prepares data, checks it and trains according to what is saved and whether a GPU is present. `RESUME_SESSION = "auto"` continues the latest unfinished training run; changing any setting below needs a new run, so then set `RESUME_SESSION = ""`. `CODE_REF` chooses the code for a new run (a branch, tag or commit). `LIDC_ROOT` may point to an existing LIDC-IDRI download in Drive (NBIA folder layout).


In [ ]:
PROJECT = "/content/drive/MyDrive/Research/LUNG-CT-SCAN"
REPOSITORY = "https://github.com/Richmiz/Risk-Aware-Sparse-Volumetric-Transformer-for-Tampered-Lung-CT-Detection.git"
CODE_REF = "main"          # code for a new run; a resumed run keeps the commit it started with
RESUME_SESSION = "auto"    # "auto" = continue the latest unfinished run; "" = new run; or a run folder
GENERATORS = ["cycle", "pix2pix", "diffusion"]  # remove one to hold it out
PATIENTS_TRAIN = 200       # of 488 publisher training patients
PATIENTS_VALIDATION = 100  # all publisher validation patients
EPOCHS = 30
LEARNING_RATE = 0.0002
EARLY_STOPPING_PATIENCE = 8
SEED = 13
LIDC_ROOT = ""             # optional existing LIDC-IDRI folder in Drive
SHUTDOWN_WHEN_DONE = True  # release the runtime after the final save check, so it stops using compute units


## 2. Connect Google Drive

Why: the dataset store and the results live here. Use the Google account containing Research/LUNG-CT-SCAN.


In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)


## 3. Get the project code from GitHub

Why: the implementation lives in the repository, not in long cells. This finds the run to continue (if any), clones the repository into the runtime, checks out its code and installs the two missing packages. The repository is public, so no sign-in is needed. Expected output: the exact commit being run.


In [ ]:
import importlib.util, json, os, subprocess, sys
from pathlib import Path
CODE_ROOT = Path("/content/rsvt-code")
env = dict(os.environ, GIT_TERMINAL_PROMPT="0")

# A resumed run keeps the code commit it started with.
RESUME_FOLDER, ref = "", CODE_REF
if RESUME_SESSION == "auto":
    for record in sorted(Path(PROJECT, "outputs/rsvt_training").glob("c2f_*/notebook_session.json"), reverse=True):
        run = record.parent
        if (run / "training/last.pt").exists() and not (run / "training/summary.json").exists():
            RESUME_FOLDER = str(run)
            break
elif RESUME_SESSION:
    RESUME_FOLDER = RESUME_SESSION
if RESUME_FOLDER:
    record = json.loads(Path(RESUME_FOLDER, "notebook_session.json").read_text(encoding="utf-8"))
    ref = record.get("code_commit") or CODE_REF
    print("Continuing the unfinished run", RESUME_FOLDER)

def git(*args):
    done = subprocess.run(["git", *args], env=env, capture_output=True, text=True)
    if done.returncode:
        raise RuntimeError(f"git {args[0]} failed: {done.stderr.strip()[-300:]}")
    return done.stdout.strip()

if not (CODE_ROOT / ".git").exists():
    git("clone", "--quiet", REPOSITORY, str(CODE_ROOT))
git("-C", str(CODE_ROOT), "fetch", "--quiet", "--tags", "origin")
commit = None
for candidate in (f"origin/{ref}", ref):
    try:
        commit = git("-C", str(CODE_ROOT), "rev-parse", "--verify", "--quiet", candidate + "^{commit}")
        break
    except RuntimeError:
        pass
if commit is None:
    raise RuntimeError(f"{ref!r} is not a branch, tag or commit of the repository")
if "rsvt" in sys.modules and globals().get("RSVT_COMMIT") not in (None, commit):
    raise RuntimeError("This session already loaded other code. Runtime > Restart session, then Run all.")
git("-C", str(CODE_ROOT), "checkout", "--quiet", "--detach", commit)
RSVT_COMMIT = commit
for module, requirement in (("pydicom", "pydicom==3.0.2"), ("SimpleITK", "SimpleITK==2.5.7")):
    if importlib.util.find_spec(module) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", requirement], check=True)
if str(CODE_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(CODE_ROOT / "src"))
print("Project code at commit", commit)


## 4. Open the run folder

Why: each run has its own Drive folder with its settings and code commit; an unfinished run is reopened. The M3Dsynth metadata tables are downloaded from the authors' GitHub at a pinned commit and checked against their recorded checksums. Expected output: the run folder in Drive.


In [ ]:
from rsvt.training.m3d_notebook import create_session
session = create_session(PROJECT, CODE_ROOT, epochs=EPOCHS, learning_rate=LEARNING_RATE,
    patience=EARLY_STOPPING_PATIENCE, seed=SEED, patients_train=PATIENTS_TRAIN,
    patients_validation=PATIENTS_VALIDATION, generators=tuple(GENERATORS),
    lidc_root=LIDC_ROOT or None, resume_session=RESUME_FOLDER, code_commit=RSVT_COMMIT)
print("Run folder in Drive:", session.root)


## 5. Check the actual runtime and GPU

Why: a server label is not proof of a GPU. This checks what PyTorch sees. Data preparation and the data checks run on CPU; training needs a GPU.


In [ ]:
runtime_information = session.show_runtime()


## 6. Review the data plan

Why: see patients, series, archive sizes, Drive storage, the state of the saved dataset and free space before any transfer. Nothing is downloaded here.


In [ ]:
data_plan = session.plan_data()


## 7. Prepare and verify M3Dsynth

Why: build the dataset store, or confirm the saved one. It downloads only what the saved dataset is missing (on the first run, everything) and otherwise reads the saved store back against its checksums. It checks free space first and stops before downloading if Google Drive or the runtime disk is too small. Each manipulated slice is compared with the real LIDC slice while it streams; a sample is kept only if every change lies inside its stored box and the author mask is complete. Expected output: progress, then a checksum-verified manifest.


In [ ]:
manifest = session.prepare_data()


## 8. Inspect the dataset with real CT slices

Why: confirm what the model will see. Expected output: counts per partition/generator/operation and training examples showing the authentic slice, the manipulated slice, their difference and the changed voxels. No test data is opened.


In [ ]:
if session.has_store():
    dataset_overview = session.show_dataset()
    preview_examples = session.show_previews()


## 9. Pre-training checks: resolution and shortcuts

Why: test the review findings on real data before spending GPU time. (a) A small CNN separates manipulated crops from authentic sites that look alike (removals vs healthy-tissue TB sites, insertions vs real-nodule TM sites) at native resolution and degraded to the 96³ voxel size; results are shown per operation. This tests the evidence at a known site, not whether the router can find it. (b) Archive identity outside the edit, mask coverage, and whether whole-scan statistics alone separate classes. Uses train/validation only. The checks run once per dataset and are reused afterwards. Training is held back if most samples failed archive verification or if whole-scan statistics alone separate the classes.


In [ ]:
diagnostics = session.run_diagnostics() if session.has_store() else None


## 10. Understand the training configuration

Why: see batches, crops, loss and risk weights, early stopping and checkpoint selection before training.


In [ ]:
configuration_table = session.show_configuration()


## 11. Build and inspect the coarse-to-fine RSVT

Why: see parameters and compute. The 96³ view gives anatomy context and a router chooses K native-resolution crops; evidence is read only there. The table compares this with running the same encoder over the whole native volume.


In [ ]:
if session.has_store():
    parameter_count = session.show_model()


## 12. Train and watch the epoch results

Why: learn from training volumes and monitor validation volumes. On a GPU runtime this trains, or continues the unfinished run; on a CPU runtime it stops here with a message. Expected output: batch progress, an updating epoch table and learning curves. The first epochs train without tail risk; the checkpoint is chosen among risk-stage epochs by the validation loss averaged over authentic, removal and insertion volumes. Where a volume was edited is taken from its changed voxels (manipulated vs real), not the author mask, which misses most of each removal. Checkpoints and logs save to Drive after every epoch.


In [ ]:
training_summary = session.train_when_ready(diagnostics) if session.has_store() else None


## 13. Review the learning curves

Why: compare training objective with validation loss and AUROCs. Falling training loss alone does not prove generalization.


In [ ]:
epoch_history = session.show_history()


## 14. Inspect the selected model’s results

Why: judge detection where it matters: removal (FB) and insertion (FM) against authentic scans, per generator, router recall (did the chosen crops contain the edit?) and overlap with the changed voxels, then how those labels compare with the author mask. Operating points are exploratory.


In [ ]:
selected_model_results = session.show_results()


## 15. Understand the held-out test set

Why: keep an independent final check. Test patients are not acquired or opened by this notebook.


In [ ]:
session.show_test_status()


## 16. Verify that the results are saved

Why: confirm Drive storage by reading files back against checksums. Expected output: **Saved and verified** and the session path.


In [ ]:
persistence_receipt = session.verify_saved()
print("Run folder:", session.root)


## 17. Release the runtime

Why: an idle runtime keeps using compute units until Colab disconnects it. This runs only after section 16 has read every saved file back, so nothing is lost; the outputs above stay visible. If an earlier cell failed, Run all stopped there and the runtime is kept so you can see the error. Set `SHUTDOWN_WHEN_DONE = False` to keep it.


In [ ]:
if SHUTDOWN_WHEN_DONE:
    try:
        from google.colab import runtime
    except ImportError:
        print("Not running in Colab; nothing to release.")
    else:
        print("Everything is saved and verified in Drive. Releasing the runtime now.")
        runtime.unassign()
else:
    print("Runtime kept (SHUTDOWN_WHEN_DONE = False). Disconnect it when done to stop using compute units.")
